<a href="https://colab.research.google.com/github/hidarihizi/Trainee/blob/main/%E3%82%A4%E3%83%B3%E3%82%BF%E3%83%BC%E3%83%B3_%E5%AE%8C%E6%88%90%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =============================================================================
# プロジェクト名：NTT株価「安値ハンティングAI」開発
# 概要：余計な出力を全てカットし、結果（改善額）のみを表示する
# =============================================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings

# 警告を非表示
warnings.filterwarnings('ignore')

# -----------------------------------------------------
# 1. データ読み込みと前処理
# -----------------------------------------------------
try:
    df = pd.read_csv('stock_price.csv')
except FileNotFoundError:
    print("❌ 'stock_price.csv' が見つかりません。")
    raise

# 日付変換とソート
df['日付'] = pd.to_datetime(df['日付'])
df = df.sort_values('日付').reset_index(drop=True)

# 出来高のクリーニング関数
def clean_volume_robust(x):
    if isinstance(x, (int, float)): return x
    x = str(x).upper()
    if 'M' in x: return float(x.replace('M', '')) * 1_000_000
    elif 'K' in x: return float(x.replace('K', '')) * 1_000
    elif 'B' in x: return float(x.replace('B', '')) * 1_000_000_000
    return float(x)

df['出来高'] = df['出来高'].apply(clean_volume_robust)
df['Day'] = df['日付'].dt.day
df['Month_ID'] = df['日付'].dt.to_period('M')

# -----------------------------------------------------
# 2. 特徴量エンジニアリング
# -----------------------------------------------------
# (1) 移動平均乖離率 (MA_Gap)
df['MA25'] = df['終値'].rolling(window=25).mean()
df['MA_Gap'] = (df['終値'] - df['MA25']) / df['MA25'] * 100

# (2) RSI (相対力指数)
delta = df['終値'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# (3) 出来高倍率 (Volume_Ratio)
df['Vol_MA15'] = df['出来高'].rolling(window=15).mean()
df['Volume_Ratio'] = df['出来高'] / df['Vol_MA15']

# 欠損値除去
df = df.dropna().reset_index(drop=True)

# -----------------------------------------------------
# 3. 正解ラベルの作成と分割
# -----------------------------------------------------
def set_target(x):
    criterion = x['終値'].quantile(0.3)
    return (x['終値'] <= criterion).astype(int)

df['Target'] = df.groupby('Month_ID', group_keys=False).apply(set_target)

test_start_date = '2023-01-01'
train = df[df['日付'] < test_start_date]
test = df[df['日付'] >= test_start_date].copy()

features = ['RSI', 'MA_Gap', 'Volume_Ratio', 'Day']
X_train = train[features]
y_train = train['Target']
X_test = test[features]
y_test = test['Target']

# -----------------------------------------------------
# 4. モデル学習 (Greedy Mode)
# -----------------------------------------------------
final_model = lgb.LGBMClassifier(
    random_state=42,
    n_estimators=1000,
    max_depth=-1,
    num_leaves=63,
    learning_rate=0.05,
    min_child_samples=5,
    verbose=-1
)

final_model.fit(X_train, y_train)
probs = final_model.predict_proba(X_test)[:, 1]
test['Probability'] = probs

# -----------------------------------------------------
# 5. 最適k値の全探索 & 結果計算
# -----------------------------------------------------
best_k = 0
max_profit = -100

# 0.50 から 0.90 まで 0.01 刻みでスキャン
for k in np.arange(0.50, 0.90, 0.01):
    temp_action = (test['Probability'] >= k).astype(int)
    temp_buys = test[temp_action == 1]

    if len(temp_buys) > 5:
        p_sum = 0
        v_months = temp_buys['Month_ID'].unique()
        for m in v_months:
            ai_p = temp_buys[temp_buys['Month_ID'] == m]['終値'].mean()
            sato_data = test[(test['Month_ID'] == m) & (test['Day'] >= 25)]
            if len(sato_data) > 0: sato_p = sato_data.iloc[0]['終値']
            else: sato_p = test[test['Month_ID'] == m].iloc[-1]['終値']
            p_sum += (sato_p - ai_p)

        avg_p = p_sum / len(v_months)

        if avg_p > max_profit:
            max_profit = avg_p
            best_k = k

# -----------------------------------------------------
# 6. 結果のみ出力
# -----------------------------------------------------
sato_avg_p = 96.13  # 基準価格
improvement_rate = (max_profit / sato_avg_p) * 100

print(f"改善幅: +{max_profit:.2f} 円")
print(f"改善率: {improvement_rate:.2f} %")

改善幅: +2.39 円
改善率: 2.49 %
